# EventDQN 24h Maximum Parallel Training on Google Colab

Notebook này chạy train-only với cấu hình mặc định tối đa của comparison pipeline:

- 5 seed chạy song song
- 2 biến thể chạy song song cho mỗi seed: `dqn_no_event` và `event_dqn`
- 5 SUMO env cho mỗi tiến trình train
- `sumo.end_time = 86400` cho mỗi episode
- Tải cực đại: **50 SUMO instance** và **10 PyTorch training process**

Colab có thể hết RAM, CPU hoặc GPU ở mức tải này. Nếu cần, giảm `SEED_WORKERS`, `VARIANT_WORKERS` hoặc `NUM_ENVS` trong cell cấu hình.

## 1. Kiểm tra runtime

Chọn `Runtime > Change runtime type > GPU` trước khi chạy.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
print('CPU count:', os.cpu_count())
print('RAM total (GB):', round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3, 2))
subprocess.run(['nvidia-smi'], check=False)


## 2. Clone hoặc cập nhật repository

In [ ]:
REPO_URL = 'https://github.com/TrongHoaP/EventDQNRL.git'
REPO_DIR = Path('/content/EventDQNRL')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())


## 3. Cài SUMO và Python dependencies

Colab đã có PyTorch phù hợp với GPU runtime. Cell này giữ PyTorch hiện có và cài các dependency còn lại từ `requirements.txt`.

In [ ]:
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'sumo', 'sumo-tools'], check=True)

requirements = []
for line in Path('requirements.txt').read_text(encoding='utf-8').splitlines():
    stripped = line.strip()
    if not stripped or stripped.startswith('#'):
        continue
    package = stripped.split('==', 1)[0].lower()
    if package not in {'torch', 'torchvision', 'torchaudio'}:
        requirements.append(stripped)

filtered_requirements = Path('/content/requirements_colab_no_torch.txt')
filtered_requirements.write_text('\n'.join(requirements) + '\n', encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(filtered_requirements)], check=True)

import torch
import traci
import sumolib

print('SUMO binary:', shutil.which('sumo'))
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 4. Xác minh pipeline hỗ trợ train song song

Repository phải chứa phiên bản `compare_pipeline` có `--seed-workers`, `--variant-workers` và `--num-envs`.

In [ ]:
help_result = subprocess.run(
    [sys.executable, '-m', 'src.rl_traffic.runners.compare_pipeline', '--help'],
    check=True,
    text=True,
    capture_output=True,
)
print(help_result.stdout)
required_options = ('--seed-workers', '--variant-workers', '--num-envs')
missing = [option for option in required_options if option not in help_result.stdout]
if missing:
    raise RuntimeError(f'Repository chưa có pipeline song song, thiếu: {missing}')


## 5. Tạo config train 24h additive

Hai config mới được tạo trong `src/configs/` từ config gốc. Config gốc không bị ghi đè.

In [ ]:
import json

MAX_STEPS = 86400
CONFIG_PAIRS = {
    'src/configs/dqn_no_event.json': 'src/configs/colab_dqn_no_event_24h.json',
    'src/configs/event_dqn.json': 'src/configs/colab_event_dqn_24h.json',
}

for source_name, output_name in CONFIG_PAIRS.items():
    source_path = Path(source_name)
    output_path = Path(output_name)
    config = json.loads(source_path.read_text(encoding='utf-8'))
    config['sumo']['end_time'] = MAX_STEPS
    config['sumo']['gui'] = False
    config['training']['device'] = 'auto'
    output_path.write_text(json.dumps(config, indent=2) + '\n', encoding='utf-8')
    print(f'Wrote {output_path}: end_time={config["sumo"]["end_time"]}, device={config["training"]["device"]}')


## 6. Cấu hình run tối đa

Mặc định bên dưới chạy đủ 100 episode cho mỗi seed và mỗi biến thể. Đổi `RUN_GROUP` khi muốn tạo run mới thay vì resume run cũ.

In [ ]:
from datetime import datetime

RUN_GROUP = f'colab_event_dqn_24h_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
SEEDS = [7, 17, 27, 37, 47]
TRAIN_EPISODES = 100
SEED_WORKERS = 5
VARIANT_WORKERS = 2
NUM_ENVS = 5

MAX_TRAIN_PROCESSES = min(SEED_WORKERS, len(SEEDS)) * min(VARIANT_WORKERS, 2)
MAX_SUMO_INSTANCES = MAX_TRAIN_PROCESSES * NUM_ENVS

print('Run group:', RUN_GROUP)
print('Seeds:', SEEDS)
print('Train episodes per seed/variant:', TRAIN_EPISODES)
print('SUMO end_time per episode:', MAX_STEPS)
print('Maximum PyTorch training processes:', MAX_TRAIN_PROCESSES)
print('Maximum SUMO instances:', MAX_SUMO_INSTANCES)
if MAX_SUMO_INSTANCES >= 50:
    print('WARNING: Mức tải này có thể vượt tài nguyên Colab. Giảm worker/env nếu runtime bị OOM hoặc disconnect.')


## 7. Chạy train-only

Cell này không chạy eval hoặc baseline. Output được ghi vào `runs/<RUN_GROUP>/`.

In [ ]:
command = [
    sys.executable,
    '-m', 'src.rl_traffic.runners.compare_pipeline',
    '--run-group', RUN_GROUP,
    '--seeds', *[str(seed) for seed in SEEDS],
    '--train-episodes', str(TRAIN_EPISODES),
    '--dqn-config', 'src/configs/colab_dqn_no_event_24h.json',
    '--event-dqn-config', 'src/configs/colab_event_dqn_24h.json',
    '--seed-workers', str(SEED_WORKERS),
    '--variant-workers', str(VARIANT_WORKERS),
    '--num-envs', str(NUM_ENVS),
    '--skip-eval',
    '--skip-baselines',
]

print(' '.join(command))
subprocess.run(command, check=True)


## 8. Kiểm tra artifact và nén kết quả

In [ ]:
run_dir = Path('runs') / RUN_GROUP
metrics_files = sorted(run_dir.glob('train/*/metrics/episode_metrics.csv'))
checkpoint_files = sorted(run_dir.glob('train/*/checkpoints/*.pt'))

print('Run directory:', run_dir.resolve())
print('Metrics files:', len(metrics_files))
print('Checkpoint files:', len(checkpoint_files))
for path in metrics_files:
    print(' -', path)

archive_path = shutil.make_archive(str(Path('/content') / RUN_GROUP), 'zip', root_dir=run_dir)
print('Archive:', archive_path)


## 9. Tùy chọn: lưu archive vào Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
drive_output_dir = Path('/content/drive/MyDrive/EventDQNRL/runs')
drive_output_dir.mkdir(parents=True, exist_ok=True)
drive_archive = drive_output_dir / Path(archive_path).name
shutil.copy2(archive_path, drive_archive)
print('Copied archive to:', drive_archive)
